# Linear Regression in Python

---

## 1. Introduction 

This notebook demonstrates fitting linear regression models in a Jupyter notebook using Python libraries and NHANES data. NHANES is a complex, weighted survey (with strata and clusters) that ideally requires survey-aware methods, but here its design will be ignored to illustrate regression on independent or convenience samples.

We will focus initially on regression models in which systolic [blood pressure](https://en.wikipedia.org/wiki/Blood_pressure) (SBP) is the outcome (dependent) variable. That is, we will predict a subject's SBP from other variables relating to that subject. 

SBP is an important indicator of cardiovascular health. It tends to increase with age, is greater for overweight people (i.e. people with greater body mass index or BMI), and also differs among demographic groups, for example among gender and ethnic groups. 

Here we will model SBP using linear regression because linear regression is a good default starting point for any regression analysis using a quantitative outcome variable.

---

## 2. Import Libraries and Load Data
First, we will import the necessary libraries for data manipulation, visualization, and statistical modeling. Then we will load the data.

The NHANES study encompasses multiple waves of data collection. Here we only use the 2015-2016 data. As with most data sets, there are some missing values in the NHANES files. For simplicity, we'll explicitly drop all observations with missing values in any of the key variables that we will use in this notebook. This is called *complete case analysis*.

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm

# Read the 2015-2016 wave of NHANES data
nhanes_df = pd.read_csv("./data/nhanes_2015_2016.csv")

# Drop unused columns, and drop rows with any missing values.
vars = ["BPXSY1", "RIDAGEYR", "RIAGENDR", "RIDRETH1", "DMDEDUC2", "BMXBMI", "SMQ020"]
nhanes_df = nhanes_df[vars].dropna()


---

## 2. Fitting a Basic Model
We start with a simple linear regression model with only one covariate, age, predicting SBP.  In the NHANES data, the variable [BPXSY1](https://wwwn.cdc.gov/Nchs/Nhanes/2015-2016/BPX_I.htm#BPXSY1) contains the first recorded measurement of SBP for a subject, and [RIDAGEYR](https://wwwn.cdc.gov/Nchs/Nhanes/2015-2016/DEMO_I.htm#RIDAGEYR) is the subject's age in years.  The model that is fit in the next cell expresses the expected value of SBP as a linear function of age.  The formula `BPXSY1 ~ RIDAGEYR` indicates that the variable named `BPXSY1` is the response variable in this regression analysis, and the analysis has one covariate, which is `RIDAGEYR`.

In [3]:
model = sm.OLS.from_formula("BPXSY1 ~ RIDAGEYR", data=nhanes_df)
result = model.fit()
print(result.summary())
print("="*78)

                            OLS Regression Results                            
Dep. Variable:                 BPXSY1   R-squared:                       0.207
Model:                            OLS   Adj. R-squared:                  0.207
Method:                 Least Squares   F-statistic:                     1333.
Date:                Wed, 22 Oct 2025   Prob (F-statistic):          2.09e-259
Time:                        17:55:48   Log-Likelihood:                -21530.
No. Observations:                5102   AIC:                         4.306e+04
Df Residuals:                    5100   BIC:                         4.308e+04
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept    102.0935      0.685    149.120      0.0

### Interpretting Regression Parameters

At the moment, we will focus on the center section of the output where the header row begins with **coef**.  This section contains the estimated values of the parameters of the regression model, their standard errors, and other values that are used to quantify the uncertainty in the regression parameter estimates.  Note that the regression parameters may also be referred to as *slopes* or *effects*.

This fitted model implies that when comparing two people whose ages differ by one year, the older person will on average have 0.48 units higher SBP than the younger person. This difference is statistically significant, based on the p-value shown under the column labeled __`P>|t|`__.  This means that there is strong evidence that there is a real **association** between between systolic blood pressure and age in this population.

SBP is measured in units of *millimeters of mercury*, expressed *mm/Hg*.  In order to better understand the meaning of the estimated regression parameter 0.48, we can look at the standard deviation of SBP:

In [5]:
print(nhanes_df["BPXSY1"].std())

18.486559500782416


The standard deviation of around 18.5 describes the *unexplained variation* in systolic blood pressure values.  It is large compared to the regression slope of 0.48, which describes the average difference between blood pressure values for two people whose ages differ by one year. Thus, while there is a substantial tendency for blood pressure to increase with age, there is also a great deal of variation among people with the same age -- we should not be surprised to find, say, a 40 year old with greater blood pressure than a 60 year old.

### R-squared and correlation
In the case of regression with a single independent variable, as we have here, there is a very close correspondence between the regression analysis and a Pearson correlation analysis, which we have discussed earlier in course 2. The primary summary statistic for assessing the strength of a predictive relationship in a linear regression model is the *R-squared*, which is shown to be 0.207 in the regression output above.  This means that 21% of the variation in SBP is explained by age.  Note that this value is exactly the same as the squared Pearson correlation coefficient between SBP and age, as shown below.

In [6]:
cc = nhanes_df[["BPXSY1", "RIDAGEYR"]].corr()
print(cc.BPXSY1.RIDAGEYR**2)

0.2071545962518702


There is a second way to interpret the R-squared, which makes use
of the *fitted values* of the regression.  The fitted values are
predictions of the blood pressure for each person in the data
set, based on their covariate values.  In this example, the only
covariate is age, so we are predicting each NHANES subject's
blood pressure as a function of their age.  If we calculate
the Pearson correlation coefficient between the fitted values
from the regression, and the actual SBP values, and then square
this correlation coefficient, we see
that we also get the R-squared from the regression:

In [7]:
cc = np.corrcoef(nhanes_df.BPXSY1, result.fittedvalues)
print(cc[0, 1]**2)

0.20715459625186938


Thus, we see that in a linear model fit with only one covariate,
the regression R-squared is equal to the squared Pearson
correlation between the covariate and the outcome, and is also
equal to the squared Pearson correlation between the fitted
values and the outcome.

---

## Adding a Second Variable to the Linear Model